In [2]:

from pathlib import Path

# Find the project root without hardcoding a computer-specific path
current_dir = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [current_dir, *current_dir.parents]:
    if (
        (candidate / "data" / "raw" / "ipl_json").is_dir()
        and (candidate / "notebooks").is_dir()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Expected a directory containing "
        "'notebooks' and 'data/raw/ipl_json'. "
        "Open this notebook from the IPL Analytics V2 project."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ipl_json"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)

for directory in [
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR
]:
    print(directory, "| Exists:", directory.is_dir())

assert RAW_DATA_DIR.is_dir()
assert INTERIM_DATA_DIR.is_dir()
assert PROCESSED_DATA_DIR.is_dir()

json_files = list(RAW_DATA_DIR.glob("*.json"))

print("\nJSON files discovered:", len(json_files))

assert len(json_files) > 0, "No JSON files found in the raw directory."

print("\nETL DIRECTORY VALIDATION: PASS")


Project root: D:\Git\Personal\ipl-player-performance-v2
D:\Git\Personal\ipl-player-performance-v2\data\raw\ipl_json | Exists: True
D:\Git\Personal\ipl-player-performance-v2\data\interim | Exists: True
D:\Git\Personal\ipl-player-performance-v2\data\processed | Exists: True

JSON files discovered: 1243

ETL DIRECTORY VALIDATION: PASS


In [3]:

import json
import pandas as pd
from pathlib import Path

match_records = []
read_errors = []

venue_city_mapping = {
    "Dubai International Cricket Stadium": "Dubai",
    "Sharjah Cricket Stadium": "Sharjah",
}

for json_file in sorted(RAW_DATA_DIR.glob("*.json")):

    try:
        with json_file.open("r", encoding="utf-8") as file:
            match_data = json.load(file)

        info = match_data.get("info", {})

        teams = info.get("teams", [])
        dates = info.get("dates", [])
        toss = info.get("toss", {})
        outcome = info.get("outcome", {})
        event = info.get("event", {})

        venue = info.get("venue")
        original_city = info.get("city")

        # Fill only the two verified missing-city venues
        city_was_imputed = (
            original_city is None and venue in venue_city_mapping
        )

        city = (
            venue_city_mapping[venue]
            if city_was_imputed
            else original_city
        )

        record = {
            # Match identification
            "match_id": str(json_file.stem),
            "source_file": json_file.name,

            # Match information
            "season": info.get("season"),
            "date": dates[0] if dates else None,
            "match_type": info.get("match_type"),
            "gender": info.get("gender"),
            "team_type": info.get("team_type"),
            "overs_limit": info.get("overs"),

            # Location
            "venue": venue,
            "city": city,
            "city_original": original_city,
            "city_was_imputed": city_was_imputed,

            # Teams
            "team1": teams[0] if len(teams) > 0 else None,
            "team2": teams[1] if len(teams) > 1 else None,
            "teams_json": json.dumps(teams),

            # Toss
            "toss_winner": toss.get("winner"),
            "toss_decision": toss.get("decision"),

            # Match outcome
            "winner": outcome.get("winner"),
            "result": outcome.get("result"),
            "method": outcome.get("method"),
            "eliminator": outcome.get("eliminator"),
            "bowl_out": outcome.get("bowl_out"),

            # Additional match metadata
            "player_of_match": json.dumps(
                info.get("player_of_match", [])
            ),
            "event_name": event.get("name"),
            "event_match_number": event.get("match_number"),
            "event_stage": event.get("stage"),
        }

        match_records.append(record)

    except Exception as exc:
        read_errors.append({
            "file": json_file.name,
            "error": str(exc)
        })

matches_etl_df = pd.DataFrame(match_records)

print("Match records extracted:", len(matches_etl_df))
print("JSON reading errors:", len(read_errors))

if read_errors:
    display(pd.DataFrame(read_errors))

print("\nMatch table shape:", matches_etl_df.shape)
print("\nColumns:")
print(matches_etl_df.columns.tolist())

display(matches_etl_df.head())


Match records extracted: 1243
JSON reading errors: 0

Match table shape: (1243, 26)

Columns:
['match_id', 'source_file', 'season', 'date', 'match_type', 'gender', 'team_type', 'overs_limit', 'venue', 'city', 'city_original', 'city_was_imputed', 'team1', 'team2', 'teams_json', 'toss_winner', 'toss_decision', 'winner', 'result', 'method', 'eliminator', 'bowl_out', 'player_of_match', 'event_name', 'event_match_number', 'event_stage']


,match_id,source_file,season,date,match_type,gender,team_type,overs_limit,venue,city,...,toss_decision,winner,result,method,eliminator,bowl_out,player_of_match,event_name,event_match_number,event_stage
0,1082591,1082591.json,2017,2017-04-05,T20,male,club,20,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,...,field,Sunrisers Hyderabad,NaN,NaN,NaN,None,"[""Yuvraj Singh""]",Indian Premier League,1.0,NaN
1,1082592,1082592.json,2017,2017-04-06,T20,male,club,20,Maharashtra Cricket Association Stadium,Pune,...,field,Rising Pune Supergiant,NaN,NaN,NaN,None,"[""SPD Smith""]",Indian Premier League,2.0,NaN
2,1082593,1082593.json,2017,2017-04-07,T20,male,club,20,Saurashtra Cricket Association Stadium,Rajkot,...,field,Kolkata Knight Riders,NaN,NaN,NaN,None,"[""CA Lynn""]",Indian Premier League,3.0,NaN
3,1082594,1082594.json,2017,2017-04-08,T20,male,club,20,Holkar Cricket Stadium,Indore,...,field,Kings XI Punjab,NaN,NaN,NaN,None,"[""GJ Maxwell""]",Indian Premier League,4.0,NaN
4,1082595,1082595.json,2017,2017-04-08,T20,male,club,20,M.Chinnaswamy Stadium,Bengaluru,...,bat,Royal Challengers Bangalore,NaN,NaN,NaN,None,"[""KM Jadhav""]",Indian Premier League,5.0,NaN


In [4]:

# Basic match table validation

print("Total records:", len(matches_etl_df))
print("Unique match IDs:", matches_etl_df["match_id"].nunique())

print(
    "Duplicate match IDs:",
    matches_etl_df["match_id"].duplicated().sum()
)

print(
    "Missing match IDs:",
    matches_etl_df["match_id"].isna().sum()
)

print(
    "Missing team1 values:",
    matches_etl_df["team1"].isna().sum()
)

print(
    "Missing team2 values:",
    matches_etl_df["team2"].isna().sum()
)

print(
    "Missing dates:",
    matches_etl_df["date"].isna().sum()
)

print(
    "Imputed city records:",
    matches_etl_df["city_was_imputed"].sum()
)

# Confirm every match has two teams
invalid_team_counts = matches_etl_df[
    matches_etl_df["team1"].isna()
    | matches_etl_df["team2"].isna()
]

assert len(read_errors) == 0
assert len(matches_etl_df) == 1243
assert matches_etl_df["match_id"].is_unique
assert matches_etl_df["match_id"].notna().all()
assert invalid_team_counts.empty
assert matches_etl_df["city_was_imputed"].sum() == 51

print("\nMATCH TABLE VALIDATION: PASS")


Total records: 1243
Unique match IDs: 1243
Duplicate match IDs: 0
Missing match IDs: 0
Missing team1 values: 0
Missing team2 values: 0
Missing dates: 0
Imputed city records: 51

MATCH TABLE VALIDATION: PASS


In [5]:

import json
import pandas as pd

innings_records = []
innings_read_errors = []

for json_file in sorted(RAW_DATA_DIR.glob("*.json")):

    try:
        with json_file.open("r", encoding="utf-8") as file:
            match_data = json.load(file)

        match_id = str(json_file.stem)

        for innings_number, innings in enumerate(
            match_data.get("innings", []),
            start=1
        ):

            total_runs = 0
            extras_runs = 0
            wicket_records = 0
            actual_dismissals = 0
            legal_deliveries = 0
            delivery_count = 0

            for over_data in innings.get("overs", []):

                for delivery in over_data.get("deliveries", []):

                    delivery_count += 1

                    runs = delivery.get("runs", {})
                    extras = delivery.get("extras", {})

                    total_runs += runs.get("total", 0)
                    extras_runs += sum(extras.values())

                    # Wides and no-balls are not legal deliveries
                    if "wides" not in extras and "noballs" not in extras:
                        legal_deliveries += 1

                    wickets = delivery.get("wickets", [])

                    wicket_records += len(wickets)

                    actual_dismissals += sum(
                        wicket.get("kind") != "retired hurt"
                        for wicket in wickets
                    )

            target = innings.get("target", {})

            record = {
                "match_id": match_id,
                "innings_number": innings_number,
                "batting_team": innings.get("team"),
                "is_super_over": innings.get("super_over", False),

                "target_runs": target.get("runs"),
                "target_overs": target.get("overs"),

                "overs_recorded": len(innings.get("overs", [])),
                "delivery_count": delivery_count,
                "legal_deliveries": legal_deliveries,

                "total_runs": total_runs,
                "extras_runs": extras_runs,
                "batter_runs": total_runs - extras_runs,

                "wicket_records": wicket_records,
                "actual_dismissals": actual_dismissals,
            }

            innings_records.append(record)

    except Exception as exc:
        innings_read_errors.append({
            "file": json_file.name,
            "error": str(exc)
        })

innings_etl_df = pd.DataFrame(innings_records)

print("Innings records extracted:", len(innings_etl_df))
print("JSON reading errors:", len(innings_read_errors))

if innings_read_errors:
    display(pd.DataFrame(innings_read_errors))

print("\nInnings table shape:", innings_etl_df.shape)

display(innings_etl_df.head(10))


Innings records extracted: 2514
JSON reading errors: 0

Innings table shape: (2514, 14)


,match_id,innings_number,batting_team,is_super_over,target_runs,target_overs,overs_recorded,delivery_count,legal_deliveries,total_runs,extras_runs,batter_runs,wicket_records,actual_dismissals
0,1082591,1,Sunrisers Hyderabad,False,NaN,NaN,20,125,120,207,7,200,4,4
1,1082591,2,Royal Challengers Bangalore,False,208.0,20.0,20,123,118,172,6,166,10,10
2,1082592,1,Mumbai Indians,False,NaN,NaN,20,125,120,184,8,176,8,8
3,1082592,2,Rising Pune Supergiant,False,185.0,20.0,20,122,119,187,4,183,3,3
4,1082593,1,Gujarat Lions,False,NaN,NaN,20,122,120,183,4,179,4,4
5,1082593,2,Kolkata Knight Riders,False,184.0,20.0,15,96,89,184,15,169,0,0
6,1082594,1,Rising Pune Supergiant,False,NaN,NaN,20,125,120,163,6,157,6,6
7,1082594,2,Kings XI Punjab,False,164.0,20.0,19,122,114,164,10,154,4,4
8,1082595,1,Royal Challengers Bangalore,False,NaN,NaN,20,124,120,157,6,151,8,8
9,1082595,2,Delhi Daredevils,False,158.0,20.0,20,124,120,142,5,137,9,9


In [6]:

print("Total innings records:", len(innings_etl_df))

print(
    "Missing match IDs:",
    innings_etl_df["match_id"].isna().sum()
)

print(
    "Missing batting teams:",
    innings_etl_df["batting_team"].isna().sum()
)

print(
    "Duplicate match-innings keys:",
    innings_etl_df.duplicated(
        subset=["match_id", "innings_number"]
    ).sum()
)

print(
    "Negative total runs:",
    (innings_etl_df["total_runs"] < 0).sum()
)

print(
    "Negative legal deliveries:",
    (innings_etl_df["legal_deliveries"] < 0).sum()
)

# Verify that every innings belongs to a known match
valid_match_ids = set(matches_etl_df["match_id"].astype(str))

orphan_innings = innings_etl_df[
    ~innings_etl_df["match_id"].astype(str).isin(valid_match_ids)
]

print("Innings without a matching match record:", len(orphan_innings))

assert len(innings_read_errors) == 0
assert innings_etl_df["match_id"].notna().all()
assert innings_etl_df["batting_team"].notna().all()
assert not innings_etl_df.duplicated(
    subset=["match_id", "innings_number"]
).any()
assert (innings_etl_df["total_runs"] >= 0).all()
assert (innings_etl_df["legal_deliveries"] >= 0).all()
assert orphan_innings.empty

print("\nINNINGS TABLE VALIDATION: PASS")


Total innings records: 2514
Missing match IDs: 0
Missing batting teams: 0
Duplicate match-innings keys: 0
Negative total runs: 0
Negative legal deliveries: 0
Innings without a matching match record: 0

INNINGS TABLE VALIDATION: PASS


In [7]:

import json
import pandas as pd

delivery_records = []
delivery_read_errors = []

for json_file in sorted(RAW_DATA_DIR.glob("*.json")):

    try:
        with json_file.open("r", encoding="utf-8") as file:
            match_data = json.load(file)

        match_id = str(json_file.stem)

        for innings_number, innings in enumerate(
            match_data.get("innings", []),
            start=1
        ):

            for over_data in innings.get("overs", []):

                over_number = over_data.get("over")

                for delivery_index, delivery in enumerate(
                    over_data.get("deliveries", []),
                    start=1
                ):

                    runs = delivery.get("runs", {})
                    extras = delivery.get("extras", {})
                    wickets = delivery.get("wickets", [])

                    is_wide = "wides" in extras
                    is_noball = "noballs" in extras

                    record = {
                        # Composite key components
                        "match_id": match_id,
                        "innings_number": innings_number,
                        "over_number": over_number,
                        "delivery_index_in_over": delivery_index,

                        # Players and teams
                        "batting_team": innings.get("team"),
                        "batter": delivery.get("batter"),
                        "bowler": delivery.get("bowler"),
                        "non_striker": delivery.get("non_striker"),

                        # Runs
                        "batter_runs": runs.get("batter", 0),
                        "extras_runs": runs.get("extras", 0),
                        "total_runs": runs.get("total", 0),

                        # Individual extras
                        "wides": extras.get("wides", 0),
                        "noballs": extras.get("noballs", 0),
                        "byes": extras.get("byes", 0),
                        "legbyes": extras.get("legbyes", 0),
                        "penalty": extras.get("penalty", 0),

                        # Delivery legality
                        "is_legal_delivery": not (
                            is_wide or is_noball
                        ),

                        # Wicket information
                        "wicket_count": len(wickets),
                        "wicket_kinds": json.dumps([
                            wicket.get("kind")
                            for wicket in wickets
                        ]),
                        "wickets_json": json.dumps(wickets),

                        # Preserve original extras structure
                        "extras_json": json.dumps(extras),
                    }

                    delivery_records.append(record)

    except Exception as exc:
        delivery_read_errors.append({
            "file": json_file.name,
            "error": str(exc)
        })

deliveries_etl_df = pd.DataFrame(delivery_records)

print("Delivery records extracted:", len(deliveries_etl_df))
print("JSON reading errors:", len(delivery_read_errors))

if delivery_read_errors:
    display(pd.DataFrame(delivery_read_errors))

print("\nDelivery table shape:", deliveries_etl_df.shape)
print("\nColumns:")
print(deliveries_etl_df.columns.tolist())

display(deliveries_etl_df.head())


Delivery records extracted: 295732
JSON reading errors: 0

Delivery table shape: (295732, 21)

Columns:
['match_id', 'innings_number', 'over_number', 'delivery_index_in_over', 'batting_team', 'batter', 'bowler', 'non_striker', 'batter_runs', 'extras_runs', 'total_runs', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'is_legal_delivery', 'wicket_count', 'wicket_kinds', 'wickets_json', 'extras_json']


,match_id,innings_number,over_number,delivery_index_in_over,batting_team,batter,bowler,non_striker,batter_runs,extras_runs,...,wides,noballs,byes,legbyes,penalty,is_legal_delivery,wicket_count,wicket_kinds,wickets_json,extras_json
0,1082591,1,0,1,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0,0,...,0,0,0,0,0,True,0,[],[],{}
1,1082591,1,0,2,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0,0,...,0,0,0,0,0,True,0,[],[],{}
2,1082591,1,0,3,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,4,0,...,0,0,0,0,0,True,0,[],[],{}
3,1082591,1,0,4,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0,0,...,0,0,0,0,0,True,0,[],[],{}
4,1082591,1,0,5,Sunrisers Hyderabad,DA Warner,TS Mills,S Dhawan,0,2,...,2,0,0,0,0,False,0,[],[],"{""wides"": 2}"


In [8]:

print("Total delivery records:", len(deliveries_etl_df))

print(
    "Duplicate delivery keys:",
    deliveries_etl_df.duplicated(
        subset=[
            "match_id",
            "innings_number",
            "over_number",
            "delivery_index_in_over"
        ]
    ).sum()
)

print(
    "Missing match IDs:",
    deliveries_etl_df["match_id"].isna().sum()
)

print(
    "Missing batters:",
    deliveries_etl_df["batter"].isna().sum()
)

print(
    "Missing bowlers:",
    deliveries_etl_df["bowler"].isna().sum()
)

print(
    "Negative total runs:",
    (deliveries_etl_df["total_runs"] < 0).sum()
)

print(
    "Negative batter runs:",
    (deliveries_etl_df["batter_runs"] < 0).sum()
)

# Check run reconciliation
run_mismatch = deliveries_etl_df[
    deliveries_etl_df["total_runs"]
    != (
        deliveries_etl_df["batter_runs"]
        + deliveries_etl_df["extras_runs"]
    )
]

print("Run reconciliation errors:", len(run_mismatch))

# Check legal and illegal deliveries
print(
    "Legal deliveries:",
    deliveries_etl_df["is_legal_delivery"].sum()
)

print(
    "Illegal deliveries:",
    (~deliveries_etl_df["is_legal_delivery"]).sum()
)

# Check match foreign keys
valid_match_ids = set(matches_etl_df["match_id"].astype(str))

orphan_deliveries = deliveries_etl_df[
    ~deliveries_etl_df["match_id"].isin(valid_match_ids)
]

print("Orphan deliveries:", len(orphan_deliveries))

assert len(delivery_read_errors) == 0
assert deliveries_etl_df["match_id"].notna().all()
assert deliveries_etl_df["batter"].notna().all()
assert deliveries_etl_df["bowler"].notna().all()

assert not deliveries_etl_df.duplicated(
    subset=[
        "match_id",
        "innings_number",
        "over_number",
        "delivery_index_in_over"
    ]
).any()

assert (deliveries_etl_df["total_runs"] >= 0).all()
assert (deliveries_etl_df["batter_runs"] >= 0).all()
assert run_mismatch.empty
assert orphan_deliveries.empty

print("\nDELIVERY TABLE VALIDATION: PASS")


Total delivery records: 295732
Duplicate delivery keys: 0
Missing match IDs: 0
Missing batters: 0
Missing bowlers: 0
Negative total runs: 0
Negative batter runs: 0
Run reconciliation errors: 0
Legal deliveries: 284630
Illegal deliveries: 11102
Orphan deliveries: 0

DELIVERY TABLE VALIDATION: PASS


In [9]:

import json
import pandas as pd

# Calculate actual dismissals from the detailed wicket records
deliveries_check = deliveries_etl_df.copy()

deliveries_check["actual_dismissals"] = (
    deliveries_check["wicket_kinds"].apply(
        lambda value: sum(
            kind != "retired hurt"
            for kind in json.loads(value)
        )
    )
)

# Aggregate delivery records by match and innings
delivery_innings_summary = (
    deliveries_check
    .groupby(["match_id", "innings_number"], as_index=False)
    .agg(
        delivery_count=("total_runs", "size"),
        legal_deliveries=("is_legal_delivery", "sum"),
        total_runs=("total_runs", "sum"),
        extras_runs=("extras_runs", "sum"),
        batter_runs=("batter_runs", "sum"),
        wicket_records=("wicket_count", "sum"),
        actual_dismissals=("actual_dismissals", "sum"),
    )
)

print("Aggregated innings:", len(delivery_innings_summary))

display(delivery_innings_summary.head())


Aggregated innings: 2514


,match_id,innings_number,delivery_count,legal_deliveries,total_runs,extras_runs,batter_runs,wicket_records,actual_dismissals
0,1082591,1,125,120,207,7,200,4,4
1,1082591,2,123,118,172,6,166,10,10
2,1082592,1,125,120,184,8,176,8,8
3,1082592,2,122,119,187,4,183,3,3
4,1082593,1,122,120,183,4,179,4,4


In [10]:

comparison = innings_etl_df.merge(
    delivery_innings_summary,
    on=["match_id", "innings_number"],
    how="outer",
    suffixes=("_innings", "_deliveries"),
    indicator=True,
    validate="one_to_one"
)

print("Merge status:")
print(comparison["_merge"].value_counts())

comparison_columns = [
    "delivery_count",
    "legal_deliveries",
    "total_runs",
    "extras_runs",
    "batter_runs",
    "wicket_records",
    "actual_dismissals",
]

mismatch_summary = {}

for column in comparison_columns:

    left = comparison[f"{column}_innings"]
    right = comparison[f"{column}_deliveries"]

    mismatch_summary[column] = (
        left.fillna(-1) != right.fillna(-1)
    ).sum()

print("\nMismatch counts:")
for column, count in mismatch_summary.items():
    print(f"{column}: {count}")

print("\nTotal innings records:", len(innings_etl_df))
print("Total delivery groups:", len(delivery_innings_summary))

# Inspect discrepancies if any
mismatched_rows = comparison[
    comparison["_merge"] != "both"
].copy()

for column in comparison_columns:
    mismatched_rows = pd.concat(
        [
            mismatched_rows,
            comparison[
                comparison[f"{column}_innings"].fillna(-1)
                != comparison[f"{column}_deliveries"].fillna(-1)
            ]
        ]
    ).drop_duplicates()

display(mismatched_rows.head(20))

assert (comparison["_merge"] == "both").all()
assert all(count == 0 for count in mismatch_summary.values())

print("\nCROSS-TABLE RECONCILIATION: PASS")


Merge status:
_merge
both          2514
left_only        0
right_only       0
Name: count, dtype: int64

Mismatch counts:
delivery_count: 0
legal_deliveries: 0
total_runs: 0
extras_runs: 0
batter_runs: 0
wicket_records: 0
actual_dismissals: 0

Total innings records: 2514
Total delivery groups: 2514


,match_id,innings_number,batting_team,is_super_over,target_runs,target_overs,overs_recorded,delivery_count_innings,legal_deliveries_innings,total_runs_innings,...,wicket_records_innings,actual_dismissals_innings,delivery_count_deliveries,legal_deliveries_deliveries,total_runs_deliveries,extras_runs_deliveries,batter_runs_deliveries,wicket_records_deliveries,actual_dismissals_deliveries,_merge



CROSS-TABLE RECONCILIATION: PASS


In [11]:

from pathlib import Path
import json

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

export_tables = {
    "matches": matches_etl_df,
    "innings": innings_etl_df,
    "deliveries": deliveries_etl_df,
}

export_results = {}

for table_name, dataframe in export_tables.items():

    output_path = PROCESSED_DATA_DIR / f"{table_name}.csv"

    dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8"
    )

    export_results[table_name] = {
        "file": output_path.name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "size_kb": round(output_path.stat().st_size / 1024, 2),
    }

    print(f"EXPORTED: {output_path}")
    print(f"Rows: {len(dataframe):,}")
    print(f"Columns: {len(dataframe.columns)}")
    print(f"Size: {output_path.stat().st_size / 1024:.2f} KB")
    print("-" * 50)

# Save an initial dataset manifest
manifest = {
    "project": "IPL Player Performance Analytics",
    "dataset_format": "CSV",
    "source": "Cricsheet JSON",
    "tables": export_results,
}

manifest_path = PROCESSED_DATA_DIR / "etl_manifest.json"

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=4)

print("\nManifest saved:", manifest_path)
print("\nCSV EXPORT COMPLETED")


EXPORTED: D:\Git\Personal\ipl-player-performance-v2\data\processed\matches.csv
Rows: 1,243
Columns: 26
Size: 352.56 KB
--------------------------------------------------
EXPORTED: D:\Git\Personal\ipl-player-performance-v2\data\processed\innings.csv
Rows: 2,514
Columns: 14
Size: 164.60 KB
--------------------------------------------------
EXPORTED: D:\Git\Personal\ipl-player-performance-v2\data\processed\deliveries.csv
Rows: 295,732
Columns: 21
Size: 29929.25 KB
--------------------------------------------------

Manifest saved: D:\Git\Personal\ipl-player-performance-v2\data\processed\etl_manifest.json

CSV EXPORT COMPLETED


In [12]:

# Reload the exported CSV files
matches_loaded = pd.read_csv(
    PROCESSED_DATA_DIR / "matches.csv",
    dtype={"match_id": "string"}
)

innings_loaded = pd.read_csv(
    PROCESSED_DATA_DIR / "innings.csv",
    dtype={"match_id": "string"}
)

deliveries_loaded = pd.read_csv(
    PROCESSED_DATA_DIR / "deliveries.csv",
    dtype={"match_id": "string"}
)

# Record-count validation
assert len(matches_loaded) == 1243
assert len(innings_loaded) == 2514
assert len(deliveries_loaded) == 295732

# Key validation
assert matches_loaded["match_id"].is_unique

assert not innings_loaded.duplicated(
    ["match_id", "innings_number"]
).any()

assert not deliveries_loaded.duplicated(
    [
        "match_id",
        "innings_number",
        "over_number",
        "delivery_index_in_over"
    ]
).any()

# Run-total validation
assert (
    deliveries_loaded["total_runs"].sum()
    == deliveries_etl_df["total_runs"].sum()
)

assert (
    deliveries_loaded["batter_runs"].sum()
    == deliveries_etl_df["batter_runs"].sum()
)

print("Matches loaded:", len(matches_loaded))
print("Innings loaded:", len(innings_loaded))
print("Deliveries loaded:", len(deliveries_loaded))

print("\nAll primary and composite keys validated.")
print("Delivery run totals preserved.")

print("\nCSV READ-BACK VALIDATION: PASS")


Matches loaded: 1243
Innings loaded: 2514
Deliveries loaded: 295732

All primary and composite keys validated.
Delivery run totals preserved.

CSV READ-BACK VALIDATION: PASS
